In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyshacl', 'rdflib'], check=False)
print('Dependencies installed.')

In [ ]:
import json
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from rdflib import Graph, Namespace, RDF, URIRef, BNode, Literal
from pyshacl import validate

SH = Namespace('http://www.w3.org/ns/shacl#')

print('Imports loaded.')

In [ ]:
# -- Stage 3 configuration ---------------------------------------------------
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# Input paths - UPDATE STAGE2_MANIFEST based on your actual folder name
STAGE2_MANIFEST = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage2_extraction' / 'stage2_manifest.json'
CCO_ONTOLOGY    = BASE_DIR / '1 - Foundation Layer' / 'CCO.ttl'
SHACL_SHAPES    = BASE_DIR / '1 - Foundation Layer' / 'cco_shapes.ttl'

# Output paths
OUTPUT_DIR = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage3_validation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STAGE3_MANIFEST = OUTPUT_DIR / 'stage3_manifest.json'
PASSED_CHUNKS_LIST = OUTPUT_DIR / 'passed_chunks.json'

# Validation settings
SHACL_INFERENCE = 'rdfs'
ALLOW_WARNINGS  = True

def is_pass(n_violations, n_warnings):
    return n_violations == 0

# === VERIFY ALL PATHS EXIST BEFORE RUNNING ===
assert STAGE2_MANIFEST.exists(), f'Stage 2 manifest not found: {STAGE2_MANIFEST}'
assert CCO_ONTOLOGY.exists(), f'CCO not found: {CCO_ONTOLOGY}'
assert SHACL_SHAPES.exists(), f'SHACL shapes not found: {SHACL_SHAPES}'

print('✓ All input paths verified')
print(f'Stage 2 manifest:  {STAGE2_MANIFEST}')
print(f'CCO ontology:      {CCO_ONTOLOGY}')
print(f'SHACL shapes:      {SHACL_SHAPES}')
print(f'Output dir:        {OUTPUT_DIR}')
print(f'Inference mode:    {SHACL_INFERENCE}')

In [ ]:
# -- Load inputs once ---------------------------------------------------------
# Load Stage 2 manifest
with open(STAGE2_MANIFEST, encoding='utf-8') as f:
    stage2_data = json.load(f)
print(f'Stage 2 manifest loaded: {len(stage2_data["results"])} chunks')
print(f'  Status counts: {stage2_data["status_counts"]}')

# Load CCO ontology
cco_graph = Graph()
cco_graph.parse(str(CCO_ONTOLOGY), format='turtle')
print(f'CCO ontology: {len(cco_graph)} triples')

# Load SHACL shapes
shacl_graph = Graph()
shacl_graph.parse(str(SHACL_SHAPES), format='turtle')
print(f'SHACL shapes: {len(shacl_graph)} triples')

# Inventory SHACL shapes for transparency
node_shapes = list(shacl_graph.subjects(RDF.type, SH.NodeShape))
print(f'  NodeShapes defined: {len(node_shapes)}')
for s in sorted(node_shapes, key=lambda x: str(x)):
    target = next(shacl_graph.objects(s, SH.targetClass), None)
    name = str(s).split('#')[-1] if '#' in str(s) else str(s).split('/')[-1]
    target_s = str(target).split('#')[-1] if target else '?'
    print(f'    {name:35s} targets cco:{target_s}')

In [ ]:
# -- Per-chunk validator ------------------------------------------------------

def extract_violation_details(results_graph):
    """Extract structured violation/warning details from SHACL results graph.
    
    Returns:
        list of dicts: [{severity, message, focus_node, result_path, source_shape_path}]
    """
    details = []
    for vr in results_graph.subjects(RDF.type, SH.ValidationResult):
        sev = next(results_graph.objects(vr, SH.resultSeverity), None)
        msg = next(results_graph.objects(vr, SH.resultMessage), None)
        focus = next(results_graph.objects(vr, SH.focusNode), None)
        path = next(results_graph.objects(vr, SH.resultPath), None)
        
        sev_str = 'Violation' if sev == SH.Violation else 'Warning' if sev == SH.Warning else str(sev or 'Info')
        details.append({
            'severity':    sev_str,
            'message':     str(msg) if msg else '',
            'focus_node':  str(focus) if focus else '',
            'result_path': str(path) if path else '',
        })
    return details


def validate_chunk(turtle_text, cco_graph, shacl_graph):
    """Validate a single chunk's turtle against CCO SHACL.
    
    Returns:
        dict with: conforms, n_violations, n_warnings, pass, details (list)
    """
    if not turtle_text:
        return {
            'conforms': False,
            'n_violations': -1,
            'n_warnings': 0,
            'pass': False,
            'parse_error': 'Empty turtle text',
            'details': [],
        }
    
    # Parse data graph
    data_graph = Graph()
    try:
        data_graph.parse(data=turtle_text, format='turtle')
    except Exception as e:
        return {
            'conforms': False,
            'n_violations': -1,
            'n_warnings': 0,
            'pass': False,
            'parse_error': str(e)[:200],
            'details': [],
        }
    
    # Merge data + CCO ontology so sh:targetClass can match domain subclasses
    # via rdfs:subClassOf inference (e.g. gro-au:NoticeProvisionObligation
    # matches cco:Obligation shape).
    combined = data_graph + cco_graph
    
    # Run validation
    try:
        conforms, results_graph, _ = validate(
            combined,
            shacl_graph=shacl_graph,
            inference=SHACL_INFERENCE,
            allow_warnings=ALLOW_WARNINGS,
            abort_on_first=False,
            advanced=False,
            js=False,
        )
    except Exception as e:
        return {
            'conforms': False,
            'n_violations': -1,
            'n_warnings': 0,
            'pass': False,
            'shacl_error': str(e)[:200],
            'details': [],
        }
    
    # Extract details
    details = extract_violation_details(results_graph)
    n_violations = sum(1 for d in details if d['severity'] == 'Violation')
    n_warnings   = sum(1 for d in details if d['severity'] == 'Warning')
    
    return {
        'conforms':     bool(conforms),
        'n_violations': n_violations,
        'n_warnings':   n_warnings,
        'pass':         is_pass(n_violations, n_warnings),
        'details':      details,
    }


print('Validator function defined.')

In [ ]:
# -- Run Stage 3 validation across all chunks --------------------------------
print('=' * 70)
print('STAGE 3 VALIDATION — running per-chunk SHACL validation')
print('=' * 70)

stage3_results = []
n_pass = 0
n_fail = 0

for i, s2r in enumerate(stage2_data['results'], 1):
    uid = s2r['unit_id']
    jur = s2r['jurisdiction']
    s2_status = s2r['parse_status']
    
    # Skip if Stage 2 didn't parse
    if s2_status != 'parsed':
        result = {
            'unit_id': uid,
            'jurisdiction': jur,
            's2_parse_status': s2_status,
            'pass': False,
            'skip_reason': f'Stage 2 parse_status={s2_status}',
            'n_violations': -1,
            'n_warnings': 0,
            'details': [],
        }
        stage3_results.append(result)
        n_fail += 1
        print(f'[{i:2d}/{len(stage2_data["results"])}] {uid:22s} | {jur:10s} | SKIP (Stage 2 failed)')
        continue
    
    # Validate
    v = validate_chunk(s2r['turtle_text'], cco_graph, shacl_graph)
    
    result = {
        'unit_id': uid,
        'jurisdiction': jur,
        's2_parse_status': s2_status,
        'conforms':     v['conforms'],
        'pass':         v['pass'],
        'n_violations': v['n_violations'],
        'n_warnings':   v['n_warnings'],
        'details':      v['details'],
    }
    if 'parse_error' in v: result['parse_error'] = v['parse_error']
    if 'shacl_error' in v: result['shacl_error'] = v['shacl_error']
    
    stage3_results.append(result)
    
    if v['pass']: n_pass += 1
    else:         n_fail += 1
    
    status = 'PASS' if v['pass'] else 'FAIL'
    minor = f' (minor: {v["n_warnings"]} warnings)' if v['n_warnings'] > 0 else ''
    print(f'[{i:2d}/{len(stage2_data["results"])}] {uid:22s} | {jur:10s} | {status:4s} | '
          f'V={v["n_violations"]:2d} W={v["n_warnings"]:2d}{minor}')

print()
print(f'Final: {n_pass} pass, {n_fail} fail (of {len(stage3_results)} total)')

In [ ]:
# -- Save Stage 3 manifest ----------------------------------------------------
status_counts = Counter(
    'pass' if r['pass'] else 'fail'
    for r in stage3_results
)

manifest = {
    'metadata': {
        'run_type': 'stage3_shacl_validation',
        'stage': 'stage3_validation',
        'validator': 'pyshacl',
        'shacl_shapes_source': str(SHACL_SHAPES),
        'cco_ontology_source': str(CCO_ONTOLOGY),
        'stage2_manifest_source': str(STAGE2_MANIFEST),
        'inference_mode': SHACL_INFERENCE,
        'pass_criterion': 'n_violations == 0 (warnings reported as minor_issues, do not fail)',
        'created_at': datetime.now().isoformat(),
        'total_chunks_in': len(stage2_data['results']),
        'total_chunks_validated': sum(1 for r in stage3_results if r.get('n_violations', -1) >= 0),
    },
    'status_counts': dict(status_counts),
    'results': stage3_results,
}

with open(STAGE3_MANIFEST, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f'Stage 3 manifest saved: {STAGE3_MANIFEST}')

# Save passed-chunks list separately for easy downstream filtering
passed = [r['unit_id'] for r in stage3_results if r['pass']]
failed = [r['unit_id'] for r in stage3_results if not r['pass']]

passed_payload = {
    'metadata': {
        'created_at': datetime.now().isoformat(),
        'source_manifest': str(STAGE3_MANIFEST),
        'pass_criterion': 'n_violations == 0',
    },
    'n_passed': len(passed),
    'n_failed': len(failed),
    'passed_unit_ids': passed,
    'failed_unit_ids': failed,
}
with open(PASSED_CHUNKS_LIST, 'w', encoding='utf-8') as f:
    json.dump(passed_payload, f, indent=2, ensure_ascii=False)
print(f'Passed-chunks list saved: {PASSED_CHUNKS_LIST}')
print(f'  Passed: {len(passed)} | Failed: {len(failed)}')

In [ ]:
# -- Stage 3 quality scan -----------------------------------------------------
print('=' * 70)
print('STAGE 3 QUALITY SCAN — violation patterns + jurisdiction breakdown')
print('=' * 70)

# Aggregate by jurisdiction
by_jur = defaultdict(lambda: {'pass': 0, 'fail': 0, 'violations': 0, 'warnings': 0})
for r in stage3_results:
    jur = r['jurisdiction']
    if r['pass']: by_jur[jur]['pass'] += 1
    else:         by_jur[jur]['fail'] += 1
    if r.get('n_violations', 0) > 0:
        by_jur[jur]['violations'] += r['n_violations']
    by_jur[jur]['warnings'] += r.get('n_warnings', 0)

print('\nPer-jurisdiction:')
print(f'  {"Jurisdiction":12s} {"Pass":5s} {"Fail":5s} {"Violations":11s} {"Warnings":9s}')
for jur in sorted(by_jur):
    s = by_jur[jur]
    print(f'  {jur:12s} {s["pass"]:5d} {s["fail"]:5d} {s["violations"]:11d} {s["warnings"]:9d}')

# Aggregate violation messages
print()
print('Violation message frequency:')
viol_msgs = Counter()
warn_msgs = Counter()
for r in stage3_results:
    for d in r.get('details', []):
        if d['severity'] == 'Violation':
            viol_msgs[d['message']] += 1
        elif d['severity'] == 'Warning':
            warn_msgs[d['message']] += 1

if viol_msgs:
    for msg, count in viol_msgs.most_common():
        print(f'  ({count}x) {msg[:90]}')
else:
    print('  (none)')

print()
print('Warning message frequency:')
if warn_msgs:
    for msg, count in warn_msgs.most_common():
        print(f'  ({count}x) {msg[:90]}')
else:
    print('  (none)')

# Failed chunks listing
failed_chunks = [r for r in stage3_results if not r['pass']]
if failed_chunks:
    print()
    print(f'Failed chunks ({len(failed_chunks)}) — excluded from downstream:')
    for r in failed_chunks:
        if 'skip_reason' in r:
            print(f'  {r["unit_id"]:22s} | {r["jurisdiction"]:10s} | {r["skip_reason"]}')
        else:
            print(f'  {r["unit_id"]:22s} | {r["jurisdiction"]:10s} | V={r["n_violations"]} W={r["n_warnings"]}')
            for d in r.get('details', []):
                if d['severity'] == 'Violation':
                    msg_short = d['message'][:80]
                    print(f'    - [{d["severity"]}] {msg_short}')

print()
print('=' * 70)
print(f'SUMMARY')
print('=' * 70)
print(f'  Total Stage 2 chunks: {len(stage2_data["results"])}')
print(f'  Validated (Stage 2 parsed): {sum(1 for r in stage3_results if r.get("n_violations", -1) >= 0)}')
print(f'  Passed (V=0):         {len([r for r in stage3_results if r["pass"]])}')
print(f'  Failed (V>0):         {len([r for r in stage3_results if not r["pass"]])}')
print(f'  Total Violations:     {sum(viol_msgs.values())}')
print(f'  Total Warnings:       {sum(warn_msgs.values())}')

In [ ]:
# -- Stage 3 SHACL descriptive metrics ---------------------------------------
# Descriptive metrics that do NOT require manual ground truth annotation.
# These characterise the SHACL validation output as a quality signal on
# the Stage 2 extraction pipeline.
#
# NOTE: These are NOT precision/recall/F1 (which would require manual labels).
# Rigorous P/R/F1 evaluation is deferred to Stage 6 (EFRO benchmark).

print('=' * 70)
print('STAGE 3 DESCRIPTIVE METRICS (SHACL-based, no ground truth needed)')
print('=' * 70)

total_chunks   = len(stage3_results)
validated      = [r for r in stage3_results if r.get('n_violations', -1) >= 0]
passed         = [r for r in stage3_results if r['pass']]
failed         = [r for r in stage3_results if not r['pass']]

total_violations = sum(max(0, r.get('n_violations', 0)) for r in validated)
total_warnings   = sum(r.get('n_warnings', 0) for r in validated)

# --- Top-level conformance metrics ---
print('\nConformance metrics:')
print(f'  Conformance rate           : {len(passed)}/{total_chunks} = {len(passed)/total_chunks:.1%}')
print(f'    (chunks with zero hard violations)')
print(f'  Violation rate             : {len(failed)}/{total_chunks} = {len(failed)/total_chunks:.1%}')
print(f'    (chunks with one or more hard violations)')

# --- Issue density (per-chunk averages) ---
print('\nIssue density:')
print(f'  Violations per chunk (avg) : {total_violations/total_chunks:.2f}')
print(f'  Warnings per chunk (avg)   : {total_warnings/total_chunks:.2f}')
print(f'  Issues per chunk (V+W avg) : {(total_violations+total_warnings)/total_chunks:.2f}')

# --- Severity composition ---
print('\nSeverity composition (of all SHACL findings):')
total_findings = total_violations + total_warnings
if total_findings > 0:
    print(f'  Hard violations  : {total_violations}/{total_findings} = {total_violations/total_findings:.1%}')
    print(f'  Soft warnings    : {total_warnings}/{total_findings} = {total_warnings/total_findings:.1%}')
else:
    print(f'  (no findings)')

# --- Distribution: how many chunks have 0/1/2/3+ issues ---
viol_dist = Counter()
warn_dist = Counter()
for r in validated:
    v = max(0, r.get('n_violations', 0))
    w = r.get('n_warnings', 0)
    viol_dist['0' if v == 0 else '1' if v == 1 else '2' if v == 2 else '3+'] += 1
    warn_dist['0' if w == 0 else '1' if w == 1 else '2' if w == 2 else '3+'] += 1

print('\nViolation count distribution:')
for k in ['0', '1', '2', '3+']:
    n = viol_dist.get(k, 0)
    bar = '#' * int(40 * n / max(1, total_chunks))
    print(f'  V={k:3s}: {n:3d} chunks ({n/total_chunks:5.1%}) {bar}')

print('\nWarning count distribution:')
for k in ['0', '1', '2', '3+']:
    n = warn_dist.get(k, 0)
    bar = '#' * int(40 * n / max(1, total_chunks))
    print(f'  W={k:3s}: {n:3d} chunks ({n/total_chunks:5.1%}) {bar}')

# --- Per-jurisdiction descriptive metrics ---
print('\nPer-jurisdiction conformance:')
print(f'  {"Jurisdiction":12s} {"N":>4s} {"Conform%":>10s} {"V/chunk":>9s} {"W/chunk":>9s}')
print(f'  {"-"*12} {"-"*4} {"-"*10} {"-"*9} {"-"*9}')
by_jur_metrics = defaultdict(lambda: {'n':0, 'pass':0, 'v':0, 'w':0})
for r in stage3_results:
    j = r['jurisdiction']
    by_jur_metrics[j]['n']    += 1
    by_jur_metrics[j]['pass'] += 1 if r['pass'] else 0
    by_jur_metrics[j]['v']    += max(0, r.get('n_violations', 0))
    by_jur_metrics[j]['w']    += r.get('n_warnings', 0)
for j in sorted(by_jur_metrics):
    s = by_jur_metrics[j]
    conform_pct = s['pass']/s['n'] if s['n'] else 0
    v_per   = s['v']/s['n'] if s['n'] else 0
    w_per   = s['w']/s['n'] if s['n'] else 0
    print(f'  {j:12s} {s["n"]:4d} {conform_pct:>9.1%} {v_per:>9.2f} {w_per:>9.2f}')

# --- Shape activation: which SHACL shapes fired? ---
print('\nSHACL shape activation (which CCO shapes fired on the data):')
shape_activations = Counter()
for r in stage3_results:
    for d in r.get('details', []):
        # Extract shape name from message (heuristic; pyshacl source shape URIs are blank nodes)
        msg = d.get('message', '')
        # Map message text to shape category
        if 'Regulation' in msg: shape_activations[('RegulationShape', d['severity'])] += 1
        elif 'Norm' in msg:     shape_activations[('NormShape', d['severity'])] += 1
        elif 'Obligation' in msg: shape_activations[('ObligationShape', d['severity'])] += 1
        elif 'Permission' in msg: shape_activations[('PermissionShape', d['severity'])] += 1
        elif 'Prohibition' in msg: shape_activations[('ProhibitionShape', d['severity'])] += 1
        elif 'Exception' in msg: shape_activations[('ExceptionShape', d['severity'])] += 1
        elif 'RoleHolding' in msg: shape_activations[('RoleHoldingShape', d['severity'])] += 1
        elif 'RegulatoryAuthorityAgent' in msg: shape_activations[('RegulatoryAuthorityAgentShape', d['severity'])] += 1
        elif 'Condition' in msg: shape_activations[('ConditionShape', d['severity'])] += 1
        else: shape_activations[('Other', d['severity'])] += 1

if shape_activations:
    for (shape, sev), count in sorted(shape_activations.items()):
        print(f'  {shape:35s} [{sev:9s}]: {count}')
else:
    print('  (no shapes fired)')

# --- Persist metrics into manifest ---
descriptive_metrics = {
    'conformance_rate':       len(passed) / total_chunks if total_chunks else 0,
    'violation_rate':         len(failed) / total_chunks if total_chunks else 0,
    'violations_per_chunk':   total_violations / total_chunks if total_chunks else 0,
    'warnings_per_chunk':     total_warnings / total_chunks if total_chunks else 0,
    'total_violations':       total_violations,
    'total_warnings':         total_warnings,
    'severity_violation_pct': total_violations / total_findings if total_findings else 0,
    'severity_warning_pct':   total_warnings   / total_findings if total_findings else 0,
    'per_jurisdiction': {
        j: {
            'n':                s['n'],
            'pass':             s['pass'],
            'conformance_rate': s['pass']/s['n'] if s['n'] else 0,
            'violations_per_chunk': s['v']/s['n'] if s['n'] else 0,
            'warnings_per_chunk':   s['w']/s['n'] if s['n'] else 0,
        }
        for j, s in by_jur_metrics.items()
    },
}

# Re-save manifest with metrics block included
manifest['descriptive_metrics'] = descriptive_metrics
with open(STAGE3_MANIFEST, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f'\nDescriptive metrics saved to manifest: {STAGE3_MANIFEST}')

print()
print('-' * 70)
print('NOTE: These descriptive metrics characterise SHACL output. They are')
print('NOT precision/recall/F1 (which would require manual ground-truth labels).')
print('Rigorous P/R/F1 evaluation is in Stage 6 (EFRO benchmark comparison).')


In [ ]:
# Cell: Diagnose violation sources
from collections import defaultdict

print('=' * 70)
print('VIOLATION SOURCE DIAGNOSIS')
print('=' * 70)

# Group violations by shape and show example focus nodes
violation_examples = defaultdict(list)

for r in stage3_results:
    if not r['pass']:
        for d in r.get('details', []):
            if d['severity'] == 'Violation':
                msg = d['message']
                # Map to shape
                if 'Regulation' in msg: shape = 'RegulationShape'
                elif 'RoleHolding' in msg: shape = 'RoleHoldingShape'
                elif 'Exception' in msg and 'modify' in msg.lower(): shape = 'Exception/modifiesNorm'
                elif 'Exception' in msg and 'condition' in msg.lower(): shape = 'Exception/hasCondition'
                elif 'Norm' in msg: shape = 'NormShape'
                else: shape = 'Other'
                
                violation_examples[shape].append({
                    'uid': r['unit_id'],
                    'jurisdiction': r['jurisdiction'],
                    'focus_node': d.get('focus_node', '')[:80],
                    'message': msg[:100]
                })

# Show top 3 examples per shape
for shape, examples in violation_examples.items():
    print(f'\n{shape} ({len(examples)} violations)')
    print('-' * 70)
    for ex in examples[:3]:
        print(f'  {ex["uid"]} | {ex["jurisdiction"]}')
        print(f'    Focus: {ex["focus_node"]}')
        print(f'    Msg:   {ex["message"]}')

In [ ]:
# Cell: Deep dive into specific failed chunks
sample_uids = ['AUS-UNIT-00033', 'AUS-UNIT-00143']

for target_uid in sample_uids:
    print('=' * 70)
    print(f'INSPECTING: {target_uid}')
    print('=' * 70)
    
    for s2r in stage2_data['results']:
        if s2r['unit_id'] == target_uid:
            print('\n--- TTL Content ---')
            print(s2r['turtle_text'])
            print('\n--- Stage 1 Decomposition ---')
            print(json.dumps(s2r.get('stage1_decomposition', {}), indent=2)[:1500])
            break
    print('\n')